# road-damage-shift — Semaine 3 : Q3, déploiement sans GPU (ONNX, INT8, latence CPU)

Trois versions du modèle S1 (YOLO11s, seed 42) : **PyTorch FP32**, **ONNX FP32**, **ONNX INT8** (quantification
statique onnxruntime, seules les convolutions, calibration sur 200 images du train source).
On mesure : taille, latence CPU du modèle seul (médiane et 90e centile sur 200 passes), latence de bout en bout,
et perte de précision avec IC bootstrap. Le coût de la quantification = INT8 − ONNX FP32 (même prétraitement).

**Réglages Kaggle** : *Accelerator* = **None** (CPU : ne consomme pas le quota GPU), Internet On,
*Add Input* = le dataset des poids S1. `SMOKE = True` d'abord (≈ 15 min), puis `SMOKE = False` via *Save Version* (≈ 1 h).

In [ ]:
# ---- Paramètres -------------------------------------------------------------
REPO_URL = "https://github.com/<TON_USER>/road-damage-shift.git"  # à remplacer
REPO_REF = "main"
REPO_DIR = "/tmp/road-damage-shift"
RAW = "/tmp/rdd_raw"
DATA_OUT = "/tmp/rdd_yolo"
SMOKE = True                 # True : 50 images par jeu d'évaluation, 20 de calibration, 30 passes de latence
S1_MANIFEST_SHA256 = "aff1da5af0e1c61885e4a59ddc058d8f659638117f99ab5d1f5df525cc57e2da"
WEIGHTS = "/kaggle/input/datasets/<TON_USER>/rds-s1-weights/yolo11s_best.pt"  # poids S1 (fichier ou dossier)

In [ ]:
!rm -rf {REPO_DIR} && git clone -q --depth 1 -b {REPO_REF} {REPO_URL} {REPO_DIR}
!pip install -q -e "{REPO_DIR}[train]" onnxslim faster-coco-eval
!nproc; lscpu | grep -E "Model name|^CPU\(s\)|Flags" | cut -c1-200

In [ ]:
import hashlib, json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

sys.path.insert(0, f"{REPO_DIR}/src")
import mlflow, onnx, onnxruntime, pandas as pd, torch, ultralytics, yaml
from ultralytics import YOLO, settings

from roadshift.deploy.bench import bench, cpu_name
from roadshift.deploy.quantize import calibration_images, quantize_int8
from roadshift.eval.compare import compare
from roadshift.eval.report import report

CFG = yaml.safe_load(open(f"{REPO_DIR}/configs/default.yaml"))
DEP, CLASSES = CFG["deploy"], CFG["data"]["classes"]
GIT_SHA = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
VERSIONS = {"torch": torch.__version__, "ultralytics": ultralytics.__version__, "onnx": onnx.__version__,
            "onnxruntime": onnxruntime.__version__, "cpu": cpu_name(), "cpu_count": os.cpu_count()}
print(GIT_SHA[:10], VERSIONS)
W = Path(WEIGHTS)
if W.is_dir():
    pts = sorted(W.rglob("*.pt")); assert len(pts) == 1, f"{W} contient {len(pts)} .pt"; W = pts[0]
W_SHA = hashlib.sha256(W.read_bytes()).hexdigest()
print("poids S1 :", W, W_SHA[:12])

## 0–1. Données (mêmes découpages qu'en S1, vérifiés par empreinte)

In [ ]:
!python -m roadshift.data.download --out {RAW} --countries Japan India Czech United_States
!python -m roadshift.data.convert --config {REPO_DIR}/configs/default.yaml --raw-root {RAW} --out {DATA_OUT} --overwrite
DATA = Path(DATA_OUT)
DATASET_INFO = json.load(open(DATA / "dataset_info.json"))
assert DATASET_INFO["manifest_sha256"] == S1_MANIFEST_SHA256, "Découpages différents de S1 : comparaison invalide"

EVAL = DATA
if SMOKE:  # mini-copie : 50 premières images de chaque liste d'évaluation, mêmes noms de fichiers
    EVAL = Path("/tmp/rdd_yolo_smoke"); shutil.rmtree(EVAL, ignore_errors=True); EVAL.mkdir()
    for d in ("images", "labels"):
        (EVAL / d).symlink_to(DATA / d)
    shutil.copy(DATA / "manifest.csv", EVAL)
    for f in DATA.glob("list_*.txt"):
        (EVAL / f.name).write_text("\n".join(f.read_text().split()[:50]) + "\n")
    for f in DATA.glob("data_*.yaml"):
        shutil.copy(f, EVAL)
print("évaluation sur :", EVAL)

## 2. Export ONNX FP32 et quantification INT8

In [ ]:
MODELS = Path("/kaggle/working/models"); MODELS.mkdir(exist_ok=True)
pt = MODELS / "yolo11s_s1.pt"; shutil.copy(W, pt)
onnx_fp32 = Path(YOLO(str(pt)).export(format="onnx", imgsz=DEP["imgsz"], opset=DEP["onnx_opset"],
                                         dynamic=False, simplify=True))
onnx_fp32 = onnx_fp32.rename(MODELS / "yolo11s_s1_fp32.onnx")
n_calib = 20 if SMOKE else DEP["int8"]["calibration_images"]
calib = calibration_images(DATA, n_calib, CFG["project"]["seed"])
t0 = time.time()
onnx_int8 = quantize_int8(onnx_fp32, MODELS / "yolo11s_s1_int8.onnx", calib, DEP["int8"], DEP["imgsz"])
QUANT_S = time.time() - t0
VARIANTS = {"pytorch_fp32": pt, "onnx_fp32": onnx_fp32, "onnx_int8": onnx_int8}
for k, p in VARIANTS.items():
    print(f"{k:13s} {p.name:26s} {p.stat().st_size / 1e6:6.1f} Mo")
print(f"quantification : {QUANT_S:.0f} s, {len(calib)} images de calibration (train source)")

## 3. Latence CPU du modèle seul (même tenseur, médiane et 90e centile)

In [ ]:
lat = DEP["latency"]
threads = min(lat["threads"], os.cpu_count())
LAT = pd.DataFrame(bench(list(VARIANTS.values()), DEP["imgsz"], threads, 5 if SMOKE else lat["warmup"],
                         30 if SMOKE else lat["runs"]))
LAT.insert(0, "variant", list(VARIANTS))
display(LAT[["variant", "size_MB", "ms_median", "ms_p90", "fps_median", "threads", "cpu"]])

## 4. Précision : évaluation des trois versions (CPU), IC bootstrap et écarts appariés
Le seuil du F1 de chaque version est choisi sur sa propre validation source. Latence de bout en bout
(prétraitement + inférence + post-traitement) relevée par Ultralytics pendant l'évaluation du test source.

In [ ]:
RUNS = Path("/kaggle/working/runs_deploy")
EVALS = {"source_val": ("data_source.yaml", "val"), "source_test": ("data_source.yaml", "test"),
         "test_United_States": ("data_eval_United_States.yaml", "val")}
E2E = []
t0 = time.time()
for v, p in VARIANTS.items():
    for tag, (yml, split) in EVALS.items():
        r = YOLO(str(p), task="detect").val(data=str(EVAL / yml), split=split, imgsz=DEP["imgsz"],
                                            batch=8 if p.suffix == ".pt" else 1, conf=CFG["eval"]["conf_map"],
                                            save_json=True, plots=False, device="cpu", verbose=False,
                                            project=str(RUNS / v), name=f"eval_{tag}", exist_ok=True)
        if tag == "source_test":
            E2E.append({"variant": v, **{f"e2e_{k}_ms": round(x, 1) for k, x in r.speed.items()}})
    print(f"{v} évalué ({(time.time() - t0) / 60:.1f} min)")
E2E = pd.DataFrame(E2E)
SCORES = pd.concat([pd.DataFrame(report(RUNS / v, EVAL, v, n_boot=CFG["eval"]["bootstrap"]["n_resamples"]))
                    for v in VARIANTS])
CMP = pd.DataFrame(compare({v: RUNS / v for v in VARIANTS}, EVAL, ["source_test", "test_United_States"],
                           "test_United_States", n_boot=CFG["eval"]["bootstrap"]["n_resamples"]))
display(SCORES[["model", "eval", "n_images", "mAP50", "mAP50_lo", "mAP50_hi", "F1", "F1_lo", "F1_hi"]])
display(CMP[CMP.comparison == "model_paired"])
display(E2E)

## 5. MLflow et sorties

In [ ]:
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for k in ("MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if (v_ := _secret(k)):
        os.environ[k] = v_
os.chdir("/kaggle/working")
mlflow.set_tracking_uri(CFG["mlflow"]["tracking_uri"] or _secret("MLFLOW_TRACKING_URI") or "sqlite:////kaggle/working/mlflow.db")
mlflow.set_experiment(CFG["mlflow"]["experiment"] + "-deploy" + ("-smoke" if SMOKE else ""))
settings.update({"mlflow": False})

OUT = Path("/kaggle/working/results"); OUT.mkdir(exist_ok=True)
suffix = "_smoke" if SMOKE else ""
LAT.to_csv(OUT / f"deploy_latency_kaggle{suffix}.csv", index=False)
SCORES.to_csv(OUT / f"deploy_scores{suffix}.csv", index=False)
CMP.to_csv(OUT / f"deploy_comparisons{suffix}.csv", index=False)
E2E.to_csv(OUT / f"deploy_e2e_latency{suffix}.csv", index=False)
with mlflow.start_run(run_name="deploy_cpu" + suffix):
    mlflow.set_tags({"stage": "S3-deploy", "smoke": SMOKE, "git_sha": GIT_SHA,
                     "dataset_sha256": DATASET_INFO["manifest_sha256"], "s1_weights_sha256": W_SHA, **VERSIONS})
    mlflow.log_params({"imgsz": DEP["imgsz"], "opset": DEP["onnx_opset"], "n_calibration": len(calib),
                       **{f"int8_{k}": str(v) for k, v in DEP["int8"].items()}, "threads": threads})
    mlflow.log_metric("quantization_seconds", QUANT_S)
    for _, r in LAT.iterrows():
        mlflow.log_metrics({f"{r.variant}/size_MB": r.size_MB, f"{r.variant}/ms_median": r.ms_median,
                            f"{r.variant}/ms_p90": r.ms_p90})
    for _, r in SCORES.iterrows():
        mlflow.log_metrics({f"{r.model}/{r['eval']}/{k}": float(r[k]) for k in ("mAP50", "mAP50_lo", "mAP50_hi", "F1")})
    for f in OUT.glob(f"deploy_*{suffix}.csv"):
        mlflow.log_artifact(str(f), artifact_path="results")
    for p in (onnx_fp32, onnx_int8):
        mlflow.log_artifact(str(p), artifact_path="models")
!cd /kaggle/working && zip -qr s3_deploy_outputs.zip results models mlflow.db mlruns 2>/dev/null; ls -lh /kaggle/working

## À me renvoyer
`s3_deploy_outputs.zip` (résultats + modèles ONNX, réutilisés en S4 pour l'API).
Pour comparer avec **ton PC**, après avoir téléchargé les modèles :
`python -m roadshift.deploy.bench --models yolo11s_s1.pt yolo11s_s1_fp32.onnx yolo11s_s1_int8.onnx --threads <nb de cœurs> --out results/deploy_latency_pc.csv`